# Uji Validitas dan Reliabilitas Instrumen Kuesioner
Mata Kuliah Statistika Probabilitas — data kuesioner **kelas 3 TI A**

Metode: validitas butir (korelasi Pearson butir–total dimensi, plus corrected item-total correlation) dan reliabilitas (Cronbach's Alpha per dimensi, plus Alpha if Item Deleted).

**Cara pakai di Google Colab:** jalankan sel dari atas ke bawah. Saat sel unggah muncul, pilih file Excel kuesioner (boleh file seluruh kelas atau file `data_kelas_3_TIA.xlsx`; program tetap memfilter kelas 3 TI A).

## A. Persiapan data

In [ ]:
import re, os, warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from google.colab import files
    DI_COLAB = True
except ImportError:          # dijalankan di komputer lokal
    DI_COLAB = False

KELAS = "3 TI A"
ALPHA_SIG = 0.05
OUT_HASIL = "." if DI_COLAB else "../hasil"
OUT_GAMBAR = "gambar" if DI_COLAB else "../gambar"
os.makedirs(OUT_GAMBAR, exist_ok=True)
os.makedirs(OUT_HASIL, exist_ok=True)

In [ ]:
if DI_COLAB:
    up = files.upload()
    nama_file = list(up.keys())[0]
else:
    nama_file = "../data/data_kelas_3_TIA.xlsx"
print("File terbaca:", nama_file)

In [ ]:
df_all = pd.read_excel(nama_file)
print("Baris pada file       :", len(df_all))
print("Distribusi kelas      :", df_all["Kelas"].value_counts().to_dict())

# Filter hanya kelas 3 TI A (data kelas lain tidak dipakai)
df = df_all[df_all["Kelas"].astype(str).str.strip() == KELAS].reset_index(drop=True)
assert df["Kelas"].astype(str).str.strip().eq(KELAS).all()

# Kolom butir dikenali dari pola judul "Nama Dimensi [pernyataan]", bukan dari posisi kolom
kolom_butir = [c for c in df.columns if "[" in c and "]" in c]
butir = df[kolom_butir]

print("Jumlah responden (n)  :", len(df))
print("Jumlah butir          :", butir.shape[1])
print("Data kosong           :", int(butir.isna().sum().sum()))

In [ ]:
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}
num = butir.apply(lambda s: s.astype(str).str.strip().map(skala)).apply(pd.to_numeric, errors="coerce")

gagal = int(num.isna().sum().sum())
print("Sel yang gagal dikonversi:", gagal)   # harus 0
assert gagal == 0, "Ada jawaban yang tidak terbaca, periksa penulisan label."
num.head()

## B. Pengelompokan butir per dimensi

In [ ]:
# Nama dimensi = teks sebelum tanda "[" (dirapikan dari spasi/titik berlebih).
dimensi = {}
for kolom in butir.columns:
    nama = re.match(r"^(.*?)\s*\[", kolom).group(1).strip().rstrip(". ").strip()
    dimensi.setdefault(nama, []).append(kolom)

# Hanya label tampilan (typo "Rsspon" pada judul kolom asli); data asli tidak diubah.
NAMA_TAMPIL = {"Rsspon emosional": "Respon Emosional"}
PREFIX = {
    "Penggunaan AI pada tugas atau projek kuliah": "PA",
    "Aspek Psikologis penggunaan AI pada tugas atau projek kuliah": "AP",
    "Kebiasaan penggunaan teknologi digital": "KD",
    "Prokrastinasi akademik": "PR",
    "Rsspon emosional": "RE",
}
dimensi = {NAMA_TAMPIL.get(k, k): v for k, v in dimensi.items()}
PREFIX = {NAMA_TAMPIL.get(k, k): v for k, v in PREFIX.items()}

kode_butir, teks_butir, daftar_butir = {}, {}, []
for nama, daftar in dimensi.items():
    for i, kolom in enumerate(daftar, 1):
        kode = f"{PREFIX[nama]}{i}"
        teks = re.search(r"\[(.*)\]", kolom).group(1).strip()
        teks = re.sub(r"^aya ", "Saya ", teks)      # perbaikan huruf awal yang terpotong
        kode_butir[kolom], teks_butir[kode] = kode, teks
        daftar_butir.append({"Dimensi": nama, "Kode Butir": kode, "Butir Soal": teks})
    print(f"{nama}: {len(daftar)} butir")
tabel_butir = pd.DataFrame(daftar_butir)
num = num.rename(columns=kode_butir)
dimensi = {nama: [kode_butir[k] for k in daftar] for nama, daftar in dimensi.items()}
print("Total butir:", sum(len(v) for v in dimensi.values()))

## C. Uji validitas (Pearson Product Moment)

In [ ]:
n = len(num)
df_bebas = n - 2
t_krit = stats.t.ppf(1 - ALPHA_SIG / 2, df_bebas)       # dua sisi
r_tabel = t_krit / np.sqrt(df_bebas + t_krit**2)
print(f"n = {n} | df = {df_bebas} | t kritis = {t_krit:.4f} | r tabel (a=0,05; dua sisi) = {r_tabel:.3f}")

hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)
    for kode in daftar:
        r, p = stats.pearsonr(x[kode], total)
        r_kor, _ = stats.pearsonr(x[kode], total - x[kode])
        hasil_validitas.append({
            "Dimensi": nama,
            "Kode Butir": kode,
            "r-hitung": round(r, 3),
            "Corrected r": round(r_kor, 3),
            "p-value": round(p, 4),
            "r-tabel": round(r_tabel, 3),
            "Keputusan": "Valid" if r > r_tabel else "Tidak valid",
            "Corrected r >= 0,30": "Ya" if r_kor >= 0.30 else "Tidak",
        })
tabel_validitas = pd.DataFrame(hasil_validitas)
tabel_validitas

In [ ]:
print("Butir valid (r-hitung > r-tabel)   :", (tabel_validitas["Keputusan"] == "Valid").sum(), "dari", len(tabel_validitas))
print("Butir lolos corrected r >= 0,30    :", (tabel_validitas["Corrected r >= 0,30"] == "Ya").sum(), "dari", len(tabel_validitas))
print("Butir corrected r > r-tabel        :", (tabel_validitas["Corrected r"] > r_tabel).sum(), "dari", len(tabel_validitas))
print()
print(tabel_validitas.groupby("Dimensi", sort=False)["Keputusan"].value_counts().unstack(fill_value=0))
print()
print("Butir tidak valid:")
print(tabel_validitas.loc[tabel_validitas["Keputusan"] == "Tidak valid", ["Dimensi", "Kode Butir", "r-hitung", "Corrected r", "p-value"]])

## D. Uji reliabilitas (Cronbach's Alpha)

In [ ]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")

hasil_reliabilitas, alpha_dihapus = [], []
for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)
    hasil_reliabilitas.append({"Dimensi": nama, "Jumlah Butir": len(daftar),
                               "Cronbach's Alpha": round(a, 3), "Kategori": kategori(a)})
    if len(daftar) > 2:
        for kode in daftar:
            a_del = cronbach_alpha(x.drop(columns=kode))
            alpha_dihapus.append({"Dimensi": nama, "Kode Butir": kode,
                                  "Alpha jika Butir Dihapus": round(a_del, 3),
                                  "Alpha Dimensi": round(a, 3),
                                  "Selisih": round(a_del - a, 3),
                                  "Efek": "Naik" if a_del > a else "Turun"})

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

In [ ]:
# Butir yang jika dihapus menaikkan alpha (hanya diamati, TIDAK dihapus otomatis)
print(tabel_alpha_dihapus[tabel_alpha_dihapus["Efek"] == "Naik"][["Dimensi", "Kode Butir", "Alpha Dimensi", "Alpha jika Butir Dihapus", "Selisih"]])

### Simulasi eksploratif (tidak mengubah hasil utama)
Butir **tidak** dihapus dari analisis utama. Sel ini hanya melihat seberapa besar alpha berubah seandainya butir yang tidak valid dikeluarkan, sebagai bahan pertimbangan perbaikan instrumen.

In [ ]:
idx = tabel_validitas.set_index("Kode Butir")
sim = []
for nama, daftar in dimensi.items():
    buruk = [k for k in daftar if idx.loc[k, "Keputusan"] == "Tidak valid"]
    if buruk:
        sisa = [k for k in daftar if k not in buruk]
        a0, a1 = cronbach_alpha(num[daftar]), cronbach_alpha(num[sisa])
        sim.append({"Dimensi": nama, "Butir Dihapus (simulasi)": ", ".join(buruk),
                    "Jumlah Butir Sisa": len(sisa), "Alpha Awal": round(a0, 3),
                    "Alpha Simulasi": round(a1, 3), "Kategori Simulasi": kategori(a1)})
tabel_simulasi = pd.DataFrame(sim)
display(tabel_simulasi)

## E. Analisis deskriptif

In [ ]:
# Skor dimensi per responden = rata-rata butir dalam dimensi (skala 1-5).
# Skewness = skewness sampel (terkoreksi); kurtosis = excess kurtosis (normal = 0).
rows = []
for nama, daftar in dimensi.items():
    s = num[daftar].mean(axis=1)
    rows.append({"Dimensi": nama, "Jumlah Butir": len(daftar), "Mean": round(s.mean(), 3),
                 "Standar Deviasi": round(s.std(ddof=1), 3), "Skewness": round(s.skew(), 3),
                 "Kurtosis": round(s.kurt(), 3), "Min": round(s.min(), 2), "Maks": round(s.max(), 2)})
tabel_deskriptif = pd.DataFrame(rows)
display(tabel_deskriptif)

# Komposisi jawaban per butir
label = {1: "STS", 2: "TS", 3: "N", 4: "S", 5: "SS"}
komposisi = []
for nama, daftar in dimensi.items():
    for kode in daftar:
        vc = num[kode].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
        row = {"Dimensi": nama, "Kode Butir": kode}
        for k in range(1, 6):
            row[f"{label[k]} (n)"] = int(vc[k])
        for k in range(1, 6):
            row[f"{label[k]} (%)"] = round(vc[k] / n * 100, 1)
        komposisi.append(row)
tabel_komposisi = pd.DataFrame(komposisi)
tabel_komposisi.head()

In [ ]:
warna = ["#b2182b", "#ef8a62", "#d9d9d9", "#67a9cf", "#2166ac"]
nama_kat = ["Sangat Tidak Setuju (STS)", "Tidak Setuju (TS)", "Netral (N)", "Setuju (S)", "Sangat Setuju (SS)"]

for no, (nama, daftar) in enumerate(dimensi.items(), 1):
    pct = pd.DataFrame({k: num[daftar].apply(lambda s: (s == k).mean() * 100) for k in range(1, 6)})
    fig, ax = plt.subplots(figsize=(9, 0.6 * len(daftar) + 2.4))
    kiri = np.zeros(len(daftar))
    for k in range(1, 6):
        v = pct[k].values
        ax.barh(daftar, v, left=kiri, color=warna[k - 1], label=nama_kat[k - 1], edgecolor="white")
        for y, (l, w) in enumerate(zip(kiri, v)):
            if w >= 7:
                ax.text(l + w / 2, y, f"{w:.0f}%", ha="center", va="center", fontsize=8,
                        color="white" if k in (1, 5) else "black")
        kiri += v
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("Persentase responden (%)")
    ax.set_ylabel("Kode butir")
    ax.set_title(f"Komposisi Jawaban Dimensi {no}: {nama}\n(n = {n} responden, kelas {KELAS})", fontsize=10)
    ax.legend(title="Kategori jawaban", loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, fontsize=8, title_fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{OUT_GAMBAR}/grafik_dimensi_{no}.png", dpi=200, bbox_inches="tight")
    plt.show()

## F. Penyimpanan hasil

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

file_hasil = f"{OUT_HASIL}/hasil_pengujian.xlsx"
with pd.ExcelWriter(file_hasil, engine="openpyxl") as w:
    df.to_excel(w, sheet_name="Data Kelas 3 TI A", index=False)
    tabel_validitas.to_excel(w, sheet_name="Hasil Validitas", index=False)
    tabel_reliabilitas.to_excel(w, sheet_name="Hasil Reliabilitas", index=False)
    tabel_alpha_dihapus.to_excel(w, sheet_name="Alpha Jika Butir Dihapus", index=False)
    tabel_deskriptif.to_excel(w, sheet_name="Statistik Deskriptif", index=False)
    tabel_komposisi.to_excel(w, sheet_name="Komposisi Jawaban", index=False)
    tabel_butir.to_excel(w, sheet_name="Daftar Butir", index=False)
    tabel_simulasi.to_excel(w, sheet_name="Simulasi Hapus Butir", index=False)
    for ws in w.book.worksheets:
        for c in ws[1]:
            c.font = Font(bold=True, name="Arial", color="FFFFFF")
            c.fill = PatternFill("solid", fgColor="1F4E79")
            c.alignment = Alignment(wrap_text=True, vertical="center")
        ws.freeze_panes = "A2"
        for i, col in enumerate(ws.columns, 1):
            lebar = max(len(str(c.value)) if c.value is not None else 0 for c in list(col)[:60])
            ws.column_dimensions[get_column_letter(i)].width = min(max(10, lebar + 2), 45)
print("Tersimpan:", file_hasil)
if DI_COLAB:
    files.download(file_hasil)

## G. Pemeriksaan konsistensi

In [ ]:
assert df["Kelas"].astype(str).str.strip().eq(KELAS).all()
assert num.shape == (len(df), sum(len(v) for v in dimensi.values()))
assert int(num.isna().sum().sum()) == 0
assert len(tabel_validitas) == num.shape[1]
print(f"OK | n = {len(df)} | butir = {num.shape[1]} | dimensi = {len(dimensi)} | r-tabel = {r_tabel:.3f}")